In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
import aplpy

import pandas as pd
import sys
from pathlib import Path
import glob

In [ ]:
catalogue_path = '/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/Bubble_Catalogue/infer_catalogue_all.csv' # 提供されたCSVファイル
catalogue_data = pd.read_csv(catalogue_path)

In [ ]:
fgn_path_all = glob.glob("/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/Galaxy_Plane/FUGIN/12CO/*.fits")
integ_path_all = glob.glob("/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/processed_fits/FUGIN/mom0/*.fits")

fgn_path_all.sort()
integ_path_all.sort()

In [ ]:
catalogue_data['GLON_center'] = (catalogue_data['ra_min'] + catalogue_data['ra_max']) / 2
catalogue_data['GLAT_center'] = (catalogue_data['dec_min'] + catalogue_data['dec_max']) / 2

In [ ]:
# 追加機能: すべてのバブルのx方向のサイズを格納するリスト
bubble_x_sizes_all = []

for i in range(len(fgn_path_all)):
    fits_path = fgn_path_all[i]
    integ_fits_path = integ_path_all[i]
    
    # 追加機能: fitsが切り替わる毎にわかりやすいメッセージを入れる
    print(f"\n{'='*50}")
    print(f"[{i+1}/{len(fgn_path_all)}] FITSファイルの処理を開始します")
    print(f"対象ファイル: {fits_path}")
    print(f"{'='*50}")

    # withブロックを使ってFITSファイルを安全に開く・閉じる
    with fits.open(integ_fits_path) as integ_hdu_list, fits.open(fits_path) as hdu_list:
        integ_hdu = integ_hdu_list[0]
        hdu = hdu_list[0]
        wcs = WCS(integ_hdu.header)
        
        raw_d = hdu.data
        header = hdu.header
        
    ny, nx = raw_d.shape[1:3]
    
    # 画像の四隅の銀河座標を取得
    glon_min, glat_min = wcs.all_pix2world(nx, 0, 0)
    glon_max, glat_max = wcs.all_pix2world(0, ny, 0)
    
    # fitsが担当する領域を調整 (領域の重なりを微調整)
    if glon_min > 10:
        glon_min += 0.5
    glon_max -= 0.5
    
    print(f"  [領域範囲] glon: {glon_min:.2f} ~ {glon_max:.2f}, glat: {glat_min:.2f} ~ {glat_max:.2f}")
    
    # 対象領域内のカタログデータを抽出
    catalogue_data_selected = catalogue_data.query(
        f"{glon_min} <= GLON_center and GLON_center <= {glon_max} and {glat_min} <= GLAT_center and GLAT_center <= {glat_max}"
    ).reset_index(drop=True)
    
    # 銀河座標 (l, b) の四隅を格納するリスト
    bubble_region_galactic = [] 
    for index, row in catalogue_data_selected.iterrows():
        b_glon_min = row['ra_min'] # カタログにはなぜかra, decの場所にglon, glatの座標が入っている
        b_glon_max = row['ra_max']
        b_glat_min = row['dec_min']
        b_glat_max = row['dec_max']
        
        # 四隅の座標をリスト化
        bubble_region_galactic.append([
            [b_glon_min, b_glat_min],
            [b_glon_max, b_glat_min],
            [b_glon_min, b_glat_max],
            [b_glon_max, b_glat_max]
        ])
    
    print(f"  [バブル検出数] {len(bubble_region_galactic)} 個")
    
    # ピクセル座標への変換
    bubble_region_pix = []
    for b_idx, bubble_region in enumerate(bubble_region_galactic):
        region_list = []
        for p_idx, world_coords in enumerate(bubble_region):
            # WCS変換 (2D配列として渡す必要があります)
            region_pix_result = wcs.wcs_world2pix([world_coords], 0)
            
            if len(region_pix_result) > 0 and len(region_pix_result[0]) == 2:
                region_list.append(region_pix_result[0])
            else:
                print(f"  [警告] バブル {b_idx}, 頂点 {p_idx} のWCS変換結果が不正です。NaNで埋めます。")
                region_list.append([np.nan, np.nan])
                
        bubble_region_pix.append(region_list)
        
    # 画像からのデータ切り出し
    cutting_ratio = 2
    cutting_data_dic = {}
    
    for bubble_num, pix_coords in enumerate(bubble_region_pix):
        # NaNを除外して最小値・最大値を計算するために np.nanmin / np.nanmax を使用
        all_x_coords = [p[0] for p in pix_coords]
        all_y_coords = [p[1] for p in pix_coords]
        
        x_min_pix = int(np.floor(np.nanmin(all_x_coords)))
        x_max_pix = int(np.ceil(np.nanmax(all_x_coords)))
        y_min_pix = int(np.floor(np.nanmin(all_y_coords)))
        y_max_pix = int(np.ceil(np.nanmax(all_y_coords)))
        
        # 中心座標と切り出し範囲の計算
        x_center = (x_min_pix + x_max_pix) // 2 
        y_center = (y_min_pix + y_max_pix) // 2
        
        x_range = ((x_max_pix - x_min_pix) // 2) * cutting_ratio
        y_range = ((y_max_pix - y_min_pix) // 2) * cutting_ratio
        
        x_min_cut = int(x_center - x_range)
        x_max_cut = int(x_center + x_range)
        y_min_cut = int(y_center - y_range)
        y_max_cut = int(y_center + y_range)
        
        # --- 追加機能: x方向のサイズをリストに格納 ---
        x_size = x_max_cut - x_min_cut
        bubble_x_sizes_all.append(x_size)
        
        # スライスのマイナスインデックスによるバグを防ぐため、0未満は0にする
        x_start_safe = max(0, x_min_cut)
        y_start_safe = max(0, y_min_cut)
        
        # 切り出し実行
        cutting_data = raw_d[:, y_start_safe:y_max_cut, x_start_safe:x_max_cut]
        
        # 辞書のキーには計算上の元の座標を保持
        pix_coord_key = (x_min_cut, x_max_cut, y_min_cut, y_max_cut)
        cutting_data_dic[pix_coord_key] = cutting_data
        
        print(f"    Bubble {bubble_num}: X range [{x_min_cut}, {x_max_cut}] (Size: {x_size}), Y range [{y_min_cut}, {y_max_cut}], Shape: {cutting_data.shape}")

# 全体の結果表示
print(f"\n{'='*50}")
print("すべてのFITSファイルの処理が完了しました。")
print(f"取得したすべてのバブルのX方向サイズリスト (計 {len(bubble_x_sizes_all)} 個):")
print(bubble_x_sizes_all)

In [ ]:
# --- 1. 統計量の計算 ---
# 上位1%の境界値（99パーセンタイル）を計算
# maxから数えて1%なので、全体の下から99%の位置
# threshold_1percent = np.percentile(bubble_x_sizes_all, 99)

# --- 3. プロットの作成 ---
plt.figure(figsize=(10, 6))
# ヒストグラムの描画
bins = range(min(bubble_x_sizes_all)//10*10, (max(bubble_x_sizes_all)//10+2)*10, 5)
n, bins_hist, patches = plt.hist(bubble_x_sizes_all, bins=bins, color='gray', edgecolor='black', alpha=0.7)

# 切り取り使用ピクセルサイズの指定(垂直線)
cut_1 = 25
cut_2 = 50
cut_3 = 100

plt.axvline(cut_1, color='red', linestyle='--', linewidth=2, label=f'pixel size : {cut_1}')
plt.axvline(cut_2, color='magenta', linestyle='--', linewidth=2, label=f'pixel size : {cut_2}')
plt.axvline(cut_3, color='blue', linestyle='--', linewidth=2, label=f'pixel size : {cut_3}')

# 装飾
plt.title('Frequency Distribution with Top 1% Line', fontsize=15)
plt.xlabel('Pixel size', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.3)
# plt.savefig("bubble_size_Histogram.png")
plt.show()